# Step 2 - Data preparation
Split the images into train / val / test (70% / 15% / 15%), then augment the train set only.

## Import libraries

In [ ]:
from pathlib import Path
import random
import shutil

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageOps
from sklearn.model_selection import train_test_split

In [ ]:
RAW_DIR = Path("../data/classification/raw")
SPLIT_DIR = Path("../data/classification/processed")
ALLOWED_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

random.seed(42)
np.random.seed(42)

## List all images with their class

In [ ]:
def get_image_paths(data_dir):
    paths = []
    labels = []
    for class_folder in sorted(data_dir.iterdir()):
        for path in class_folder.iterdir():
            if path.suffix.lower() in ALLOWED_EXTENSIONS:
                paths.append(path)
                labels.append(class_folder.name)
    return paths, labels

In [ ]:
paths, labels = get_image_paths(RAW_DIR)
print("Total images:", len(paths))

## Split 70% / 15% / 15%
`stratify` keeps the same class proportions in each set.

In [ ]:
def split_data(paths, labels):
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths, labels, test_size=0.30, stratify=labels, random_state=42
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42
    )
    return (train_paths, train_labels), (val_paths, val_labels), (test_paths, test_labels)

In [ ]:
train_data, val_data, test_data = split_data(paths, labels)
print("Train:", len(train_data[0]))
print("Val:", len(val_data[0]))
print("Test:", len(test_data[0]))

## Copy the images into folders
`processed/<split>/<class>/image.jpg`. The old folder is deleted first so we can run this again.

In [ ]:
def copy_images(paths, labels, split_name):
    for path, label in zip(paths, labels):
        folder = SPLIT_DIR / split_name / label
        folder.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, folder / path.name)

In [ ]:
shutil.rmtree(SPLIT_DIR, ignore_errors=True)

copy_images(train_data[0], train_data[1], "train")
copy_images(val_data[0], val_data[1], "val")
copy_images(test_data[0], test_data[1], "test")

## Count the images in each set

In [ ]:
def count_images(split_name):
    counts = {}
    for class_folder in sorted((SPLIT_DIR / split_name).iterdir()):
        counts[class_folder.name] = len(list(class_folder.iterdir()))
    return counts

In [ ]:
for split_name in ["train", "val", "test"]:
    counts = count_images(split_name)
    print(split_name, sum(counts.values()), counts)

## Data augmentation: blur, noise, flip

In [ ]:
def add_blur(image):
    return image.filter(ImageFilter.GaussianBlur(radius=1))


def add_noise(image):
    pixels = np.array(image).astype(np.float32)
    noise = np.random.normal(0, 10, pixels.shape)
    noisy = np.clip(pixels + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)


def flip(image):
    return ImageOps.mirror(image)

In [ ]:
def show_augmentations(path):
    image = Image.open(path).convert("RGB")
    images = [image, add_blur(image), add_noise(image), flip(image)]
    titles = ["original", "blur", "noise", "flip"]

    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.show()

In [ ]:
show_augmentations(train_data[0][0])

## Augment the train set only
Each class gets new augmented images until it has as many images as the biggest class.
This reduces the imbalance between classes. Val and test are not touched.

In [ ]:
def augment_class(class_folder, target_count):
    images = list(class_folder.iterdir())
    missing = target_count - len(images)
    augmentations = [add_blur, add_noise, flip]

    for i in range(missing):
        path = random.choice(images)
        image = Image.open(path).convert("RGB")
        augment = random.choice(augmentations)
        new_image = augment(image)
        new_image.save(class_folder / f"aug_{i}_{path.name}")

In [ ]:
train_counts = count_images("train")
target_count = max(train_counts.values())
print("Target per class:", target_count)

for class_name in train_counts:
    augment_class(SPLIT_DIR / "train" / class_name, target_count)

In [ ]:
for split_name in ["train", "val", "test"]:
    counts = count_images(split_name)
    print(split_name, sum(counts.values()), counts)